In [5]:
import pandas as pd # python software library written for data manipulation and analsis
import numpy as np # python library working with numerical array
import seaborn as ans # open source python library based on matplotlib used for data visualization and EDA
import matplotlib.pyplot as plt # graphical analysis
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report
import re
import string
import os
# import request
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import MultinomialNB

### Importing Dataset

In [11]:
data_fake = pd.read_csv('data/Fake.csv')
data_true = pd.read_csv('data/True.csv')
data_news1 = pd.read_csv('data/news.csv')
data_news2 = pd.read_csv('data/news2.csv')


In [12]:
data_news2


,text,label
0,Gere faults Trump for blurring meaning of 'ref...,1
1,German parties start to find common ground in ...,1
2,Senate Democratic leader says Attorney General...,1
3,"Tennis: Kyrgios fined $10,000 for Shanghai wal...",1
4,Trump Threw Mar-A-Lago Fundraiser For Woman A...,0
...,...,...
45752,Heavily-Armed Man Claiming To Be Jesus Arrest...,0
45753,Evangelicals Across The Spectrum Are Clarifyin...,0
45754,"Senators question Kaleo' $4,500 tag on opioid ...",1
45755,CNN Does NOT Hold Back; Completely Makes Fun ...,0


In [13]:
data_news1

,text,label
0,Donald Trump just couldn t wish all Americans ...,FAKE
1,House Intelligence Committee Chairman Devin Nu...,FAKE
2,"On Friday, it was revealed that former Milwauk...",FAKE
3,"On Christmas day, Donald Trump announced that ...",FAKE
4,Pope Francis used his annual Christmas Day mes...,FAKE
...,...,...
44893,BRUSSELS (Reuters) - NATO allies on Tuesday we...,REAL
44894,"LONDON (Reuters) - LexisNexis, a provider of l...",REAL
44895,MINSK (Reuters) - In the shadow of disused Sov...,REAL
44896,MOSCOW (Reuters) - Vatican Secretary of State ...,REAL


In [14]:
data_news1['label'] = data_news1['label'].map({'REAL': 1, 'FAKE': 0})

In [15]:
data_news1

,text,label
0,Donald Trump just couldn t wish all Americans ...,0
1,House Intelligence Committee Chairman Devin Nu...,0
2,"On Friday, it was revealed that former Milwauk...",0
3,"On Christmas day, Donald Trump announced that ...",0
4,Pope Francis used his annual Christmas Day mes...,0
...,...,...
44893,BRUSSELS (Reuters) - NATO allies on Tuesday we...,1
44894,"LONDON (Reuters) - LexisNexis, a provider of l...",1
44895,MINSK (Reuters) - In the shadow of disused Sov...,1
44896,MOSCOW (Reuters) - Vatican Secretary of State ...,1


In [16]:
data_fake.sample(5)

,title,text,subject,date
19133,IN ORDER FOR TRUMP TO “Drain The Swamp” He’s G...,President Donald Trump is a man in a hurry. Th...,left-news,"Feb 15, 2017"
11712,WATCH Huge Crowd Of Muslims Admit That ALL Mus...,https://www.youtube.com/watch?v=8Mehk5eWcZA,politics,"Feb 8, 2017"
21578,DETROIT’S AL SHARPTON WANNABE Attempts To Bull...,Never mind the fact that Kid Rock fought for a...,left-news,"Jul 10, 2015"
15843,HILLARY SHOWS HER TRUE COLORS IN A VIDEO SHE D...,Since Hillary s making a big announcement toda...,politics,"Apr 12, 2015"
1317,Comey Plans To Tell The World About Trump’s I...,Former FBI Director James Comey is now one of ...,News,"May 31, 2017"


In [17]:
data_true.sample(5)

,title,text,subject,date
10707,Senators say Republican consensus is no vote o...,WASHINGTON (Reuters) - Key senators said on Tu...,politicsNews,"February 23, 2016"
13264,"Zimbabwe truck accident kills 21, injures others",HARARE (Reuters) - At least 21 people died in ...,worldnews,"December 3, 2017"
15661,Saudi Prince Alwaleed has invested billions in...,DUBAI (Reuters) - The detention of Saudi Arabi...,worldnews,"November 5, 2017"
13685,U.N. to offer direct talks to Syrian negotiato...,GENEVA (Reuters) - Syrian government and oppos...,worldnews,"November 28, 2017"
17530,Saudi Arabia welcomes new U.S. strategy toward...,DUBAI (Reuters) - Saudi Arabia welcomed the ne...,worldnews,"October 13, 2017"


In [18]:
data_fake.shape,data_true.shape

((23481, 4), (21417, 4))

In [19]:
# # Removing last 10 rows for manual testing
# data_fake_manual_testing=data_fake.tail(10)
# for i in range(23480,23470,-1):
#     data_fake.drop([i], axis=0, inplace=True)

# data_true_manual_testing=data_true.tail(10)
# for i in range(21416,21406,-1):
#     data_true.drop([i],axis=0,inplace = True)


In [20]:
data_fake.shape,data_true.shape

((23481, 4), (21417, 4))

### Inserting a column "label" as target feature

In [21]:
data_fake['label'] = 0
data_true['label'] = 1

In [22]:
data_fake.shape,data_true.shape

((23481, 5), (21417, 5))

### Merging True and Fake Dataframes

In [23]:
data_merge = pd.concat([data_fake,data_true],axis=0)
data_merge.head()

,title,text,subject,date,label
0,Donald Trump Sends Out Embarrassing New Year’...,Donald Trump just couldn t wish all Americans ...,News,"December 31, 2017",0
1,Drunk Bragging Trump Staffer Started Russian ...,House Intelligence Committee Chairman Devin Nu...,News,"December 31, 2017",0
2,Sheriff David Clarke Becomes An Internet Joke...,"On Friday, it was revealed that former Milwauk...",News,"December 30, 2017",0
3,Trump Is So Obsessed He Even Has Obama’s Name...,"On Christmas day, Donald Trump announced that ...",News,"December 29, 2017",0
4,Pope Francis Just Called Out Donald Trump Dur...,Pope Francis used his annual Christmas Day mes...,News,"December 25, 2017",0


In [24]:
data_merge.columns

Index(['title', 'text', 'subject', 'date', 'label'], dtype='str')

### Removing columns which are not required

In [25]:
data = data_merge.drop(["title", "subject","date"], axis = 1)

In [26]:
data.isnull().sum()

text     0
label    0
dtype: int64

### Random Shuffling the dataframe

In [27]:
data = data.sample(frac = 1)

In [28]:
data.head()

,text,label
18851,Conspiracy theories have been surrounding the ...,0
297,As if Trump calling for a private organization...,0
16522,MADRID (Reuters) - Catalan separatist leader C...,1
16032,The President and First Lady touched down in W...,0
12234,LIMA (Reuters) - Peruvian President Pedro Pabl...,1


In [29]:
data.shape

(44898, 2)

In [30]:
data.reset_index(inplace = True)
data.drop(["index"], axis = 1, inplace = True)

In [31]:
data.columns

Index(['text', 'label'], dtype='str')

In [32]:
data.head()

,text,label
0,Conspiracy theories have been surrounding the ...,0
1,As if Trump calling for a private organization...,0
2,MADRID (Reuters) - Catalan separatist leader C...,1
3,The President and First Lady touched down in W...,0
4,LIMA (Reuters) - Peruvian President Pedro Pabl...,1


In [33]:
final_data= pd.concat([data,data_news1,data_news2],axis=0)
final_data.sample(10)

,text,label
39397,BELFAST (Reuters) - A senior member of Norther...,1
29895,VANCOUVER (Reuters) - British Columbia has hir...,1
43697,LONDON (Reuters) - The British government need...,1
36893,WASHINGTON (Reuters) - The son of a Russian la...,1
25378,The bipartisan congressional language creates ...,0
5445,(Reuters) - A U.S. congressman called on the D...,1
14217,"Terrorism has gripped Europe, America has two ...",0
22966,House panel probing private email use by White...,1
13223,Vice Presidential candidate Tim Kaine has some...,0
4433,The Republican Party is busy with a pitted civ...,0


In [34]:
final_data.shape

(135553, 2)

In [35]:
final_data.isnull().sum()

text     0
label    0
dtype: int64

## Creating a function to process the texts

In [36]:
def clean_text(text):
    text = str(text).lower()

    text = re.sub(r'https?://\S+', ' ', text)
    text = re.sub(r'www\.\S+', ' ', text)

    text = re.sub(r'<.*?>', ' ', text)

    text = re.sub(r'[^a-zA-Z\s]', ' ', text)

    text = re.sub(r'\s+', ' ', text)

    return text.strip()

In [37]:
final_data["text"] = final_data["text"].apply(clean_text)

### Defining dependent and independent variables

In [38]:
x = final_data["text"]
y = final_data["label"]

### Splitting Training and Testing

In [39]:
x_train, x_test, y_train, y_test = train_test_split(
    x,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

### Convert text to vectors

In [40]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorization = TfidfVectorizer(
    stop_words="english",
    ngram_range=(1,2),
    max_features=100000,
     min_df=2
)
xv_train = vectorization.fit_transform(x_train)
xv_test = vectorization.transform(x_test)

### Logistic Regression

In [41]:
from sklearn.linear_model import LogisticRegression

LR = LogisticRegression(max_iter=2000)
LR.fit(xv_train,y_train)

,"max_iter max_iter: int, default=100Maximum number of iterations taken for the solvers to converge.",2000
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary <random_state>` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is '

In [42]:
pred_lr=LR.predict(xv_test)

In [43]:
LR.score(xv_test, y_test)

0.9899302865995352

In [44]:
print(classification_report(y_test, pred_lr))

              precision    recall  f1-score   support

           0       0.99      0.99      0.99     13964
           1       0.99      0.99      0.99     13147

    accuracy                           0.99     27111
   macro avg       0.99      0.99      0.99     27111
weighted avg       0.99      0.99      0.99     27111



In [45]:
import joblib

joblib.dump(vectorization, "models/tfidf_vectorizer.pkl")
joblib.dump(LR, "models/logistic_regression.pkl")

print("Models saved successfully!")

Models saved successfully!


### Random Forest

In [46]:
from sklearn.ensemble import RandomForestClassifier

RFC = RandomForestClassifier(n_estimators=300, random_state=42)
RFC.fit(xv_train,y_train)

pred_rfc=RFC.predict(xv_test)

RFC.score(xv_test, y_test)

print(classification_report(y_test, pred_rfc))

              precision    recall  f1-score   support

           0       0.99      1.00      0.99     13964
           1       1.00      0.99      0.99     13147

    accuracy                           0.99     27111
   macro avg       0.99      0.99      0.99     27111
weighted avg       0.99      0.99      0.99     27111



### Linear SVM

In [ ]:
from sklearn.svm import LinearSVC

SVM = LinearSVC(random_state=42)
SVM.fit(xv_train,y_train)

pred_svm=SVM.predict(xv_test)

SVM.score(xv_test, y_test)

print(classification_report(y_test, pred_svm))

              precision    recall  f1-score   support

           0       0.99      1.00      1.00     13964
           1       1.00      0.99      1.00     13147

    accuracy                           1.00     27111
   macro avg       1.00      1.00      1.00     27111
weighted avg       1.00      1.00      1.00     27111



### Multinomial Naive Bayes

In [ ]:
from sklearn.naive_bayes import MultinomialNB

NB = MultinomialNB()
NB.fit(xv_train,y_train)

pred_nb=NB.predict(xv_test)

NB.score(xv_test, y_test)

print(classification_report(y_test, pred_nb))

              precision    recall  f1-score   support

           0       0.95      0.96      0.95     13964
           1       0.95      0.95      0.95     13147

    accuracy                           0.95     27111
   macro avg       0.95      0.95      0.95     27111
weighted avg       0.95      0.95      0.95     27111



### News API

In [ ]:

# import os
# API_KEY = os.getenv("NEWS_API_KEY")

### search function

In [ ]:
REAL_THRESHOLD = 0.65
FAKE_THRESHOLD = 0.65

def search_news(query):

    url = "https://newsapi.org/v2/everything"

    params = {
        "q": query,
        "language":"en",
        "sortBy":"relevancy",
        "pageSize":10,
        "apiKey":API_KEY
    }

    response = requests.get(
        url,
        params=params
    )

    if response.status_code != 200:
        return []

    data = response.json()

    return data.get(
        "articles",
        []
    )


### ML prediction

In [ ]:
# def get_ml_prediction(news):

#     news = clean_text(news)
    
#     vector = vectorization.transform([news])

#     pred = LR.predict(vector)[0]

#     proba = LR.predict_proba(vector)[0]

#     fake_prob = float(proba[0])
#     real_prob = float(proba[1])
    
#     return pred, fake_prob, real_prob

### Evidence Retrivial

In [ ]:
# def evidence_score(news, articles):

#     if len(articles) == 0:
#         return 0, None

#     docs = [news]

#     for article in articles:

#         title = article.get("title", "")

#         description = article.get(
#             "description",
#             ""
#         )

#         docs.append(
#             title + " " + description
#         )

#     temp_vectorizer = TfidfVectorizer(
#         stop_words="english"
#     )

#     matrix = temp_vectorizer.fit_transform(
#         docs
#     )

#     sims = cosine_similarity(
#         matrix[0:1],
#         matrix[1:]
#     )[0]

#     best = np.argmax(sims)

#     return float(sims[best]), articles[best]

### Decision Engine

In [ ]:
# def final_prediction(news):

#     pred, fake_prob, real_prob = get_ml_prediction(news)

#     articles = search_news(news)

#     evidence, best_article = evidence_score(
#         news,
#         articles
#     )

#     if evidence > 0.50 and real_prob > 0.70:
    
#         verdict = "Likely Real"
    
#     elif evidence < 0.10 and fake_prob > 0.70:
    
#         verdict = "Likely Fake"
    
#     elif real_prob > 0.90:
    
#         verdict = "Likely Real"
    
#     elif fake_prob > 0.90:
    
#         verdict = "Likely Fake"
    
#     else:
    
#         verdict = "Uncertain"

#     return {
#         "prediction": pred,
#         "fake_prob": fake_prob,
#         "real_prob": real_prob,
#         "evidence": evidence,
#         "best_article": best_article,
#         "verdict": verdict
#     }

### model testing 

In [ ]:
# def verify_news(news):

#     result = final_prediction(news)

#     print("\n========== ML MODEL ==========")

#     print(
#         "Fake Probability:",
#         round(result["fake_prob"],3)
#     )

#     print(
#         "Real Probability:",
#         round(result["real_prob"],3)
#     )

#     print("\n========== EVIDENCE ==========")

#     print(
#         "Similarity:",
#         round(result["evidence"],3)
#     )

#     if result["best_article"]:

#         print("\nBest Supporting Article:")

#         print(
#             result["best_article"]["title"]
#         )

#         print(
#             result["best_article"].get(
#                 "url",
#                 ""
#             )
#         )

#     print("\n========== FINAL ==========")

#     print(
#         result["verdict"]
#     )